In [1]:
from requests import get
import pandas as pd
import json
import numpy as np
import chompjs
import re

In [2]:
# tv2 kreds data findes på deres hjemmeside i nemme at huske sti webpack/repo/components-tv2/src/components/Blocks/Election/"Results/data"
with open('./TV2/greaterConstituencies.ts', 'r', encoding='utf-8') as f:
    content = f.read()

storkreds = chompjs.parse_js_object(re.search(r'=\s*(\[.*?\]);', content, re.DOTALL).group(1))

In [18]:
# Kør kun første gang
svar = {}
for kreds in storkreds:
    print(kreds)
    res = get(f"https://election-api.services.tv2.dk/fv/FV2026/candidatetest/answers/{kreds['value']}")
	# deres hjemmeside brugte denne alternative url, men den gamle url ser også ud til at virke...
    # https://election-api.services.tv2.dk/fv/FV2026_TEST1/results/candidates/[kreds]
    svar[kreds['value']] = res.json()
##gem svarerne
json.dump(svar, open('./TV2/alle_svar.json', 'w'))

{'value': 9000400, 'label': 'Bornholms Storkreds', 'councilType': 'folketing', 'councilTypePlural': 'folketinget'}
{'value': 9000600, 'label': 'Fyns Storkreds', 'councilType': 'folketing', 'councilTypePlural': 'folketinget'}
{'value': 9000200, 'label': 'Københavns Omegns Storkreds', 'councilType': 'folketing', 'councilTypePlural': 'folketinget'}
{'value': 9000100, 'label': 'Københavns Storkreds', 'councilType': 'folketing', 'councilTypePlural': 'folketinget'}
{'value': 9001000, 'label': 'Nordjyllands Storkreds', 'councilType': 'folketing', 'councilTypePlural': 'folketinget'}
{'value': 9000300, 'label': 'Nordsjællands Storkreds', 'councilType': 'folketing', 'councilTypePlural': 'folketinget'}
{'value': 9000500, 'label': 'Sjællands Storkreds', 'councilType': 'folketing', 'councilTypePlural': 'folketinget'}
{'value': 9000700, 'label': 'Sydjyllands Storkreds', 'councilType': 'folketing', 'councilTypePlural': 'folketinget'}
{'value': 9000900, 'label': 'Vestjyllands Storkreds', 'councilType'

In [3]:
tv2_spg = [f'tv2-fv26-danmark-{x+1}' for x in range(24)]

In [4]:
# smæk det hele pænt sammen
svar = json.load(open('./TV2/alle_svar.json'))
tv2_svar = {}
tv2_kandidater = {}
for kreds_id, kandidater in svar.items():
    for kandidat in kandidater:
        tv2_kandidater[kandidat['id']] = {'navn': kandidat['name'], 'kreds': kandidat['areaName'], 'job': kandidat['occupation'], 'parti': kandidat['partyName'], 'alder': kandidat['age']}
        tv2_svar[kandidat['id']] = {x: kandidat['answers'][x]['answer'] for x in tv2_spg if x in kandidat['answers']}
tv2_kandidater = pd.DataFrame(tv2_kandidater).T
tv2_svar = pd.DataFrame(tv2_svar).T

In [5]:
# norm mellem 0 og 1
ma = tv2_svar.max()
mi = tv2_svar.min()
tv2_svar = (tv2_svar-mi)/(ma-mi)

In [6]:
q = tv2_svar.copy()
q['parti'] = [tv2_kandidater.loc[x, 'parti'] for x in tv2_svar.index]
q['navn'] = [tv2_kandidater.loc[x, 'navn'] for x in tv2_svar.index]
q['job'] = [tv2_kandidater.loc[x, 'job'] for x in tv2_svar.index]
q['alder'] = [tv2_kandidater.loc[x, 'alder'] for x in tv2_svar.index]
q['kreds'] = [tv2_kandidater.loc[x, 'kreds'] for x in tv2_svar.index]
#q.parti.replace("Konservative", "Det Konservative Folkeparti", inplace=True)

In [7]:
# dejligt nemt. Tak TV2s programmører!

In [8]:
# DR's data er...mere gemt...men jeg fandt en acceptabel måde at hive det ud på via scrapy
# Men selvfølgelig har de ændret på det i år...og nu er det hele gemt i et script tag i html filen.
# kør kommando i TakeTheDR mappen:
# scrapy crawl DR -O DR_data.json -L WARNING

alt_dr = json.load(open('./TakeTheDR/DR_data.json'))

In [9]:
#lad os lige kigge på data
alt_dr[0].keys()

dict_keys(['candidate', 'candidateAnswers', 'questions'])

In [10]:
alt_dr[0]['candidate'].keys()

dict_keys(['ID', 'Firstname', 'LastName', 'Picture', 'Photographer', 'Gender', 'Birthdate', 'UrlKey', 'PersonUrlKey', 'CurrentParty', 'CurrentPartyCode', 'City', 'Zipcode', 'Facebook', 'Homepage', 'Instagram', 'Twitter', 'Profession', 'Education', 'LeisureInterest', 'PoliticsInterest', 'LinedUpForParty', 'LinedUpForPartyCode', 'LineUps'])

In [12]:
drdata = [] # tømmerne den lige for at undgå at skulle genskrive koden neden under
for kan in alt_dr:
    if kan['candidate']['LineUps'][0]['groupType'] == 'Municipality':
        drdata.append(kan)

In [16]:
alt_dr[2]['candidateAnswers'][0].keys()

dict_keys(['QuestionID', 'Answer', 'AnswerList', 'Info', 'IsImportant', 'ID_CandidateQuestionType', 'IsTopicQuestionType'])

In [17]:
alt_dr[2]['questions'][0].keys()

dict_keys(['Id', 'Title', 'Question', 'Info', 'ArgumentFor', 'ArgumentAgainst', 'OptionalTitle', 'OptionalQuestion', 'OptionalInfo', 'OptionalArgumentFor', 'OptionalArgumentAgainst', 'ID_CandidateQuestionType', 'QuestionListData', 'IsTopicQuestionType', 'RelatedQuestionId'])

In [18]:
# lad os få et overblik over spørgsmålene
sprgs = {}
n_sprg = [] # hvor mange spørgsmål er der per kandidat
for kandidat in alt_dr:
    n_sprg.append(len(kandidat['questions']))
    for sprg in kandidat['questions']:
        sprgs[sprg['Id']] = sprg['Question']

In [19]:
print(np.min(n_sprg), np.mean(n_sprg), np.median(n_sprg), np.max(n_sprg))
# ok- 25 spørgsmål til alle

25 25.0 25.0 25


In [47]:
sprgs = {f"DR{x}": y for x,y in sprgs.items()}

In [48]:
json.dump(sprgs, open('TakeTheDR/sprgs.json', 'wt'), indent=4, ensure_ascii=False)

In [23]:
bogstaver = json.load(open('various.json'))['bogstav_leg']

In [53]:
# strukturen var som sagt
# 'candidate': {'ID', 'Firstname', 'LastName', 'Picture', 'CroppedPicture', 'Photographer', 'Gender', 'Birthdate', 'UrlKey', 'PersonUrlKey', 'CurrentParty', 'CurrentPartyCode', 'City', 'Zipcode', 'Facebook', 'Homepage', 'Instagram', 'Twitter', 'Profession', 'Education', 'LeisureInterest', 'PoliticsInterest', 'LinedUpForParty', 'LinedUpForPartyCode', 'Keytopics', 'LineUps', 'OnOtherElections']}
# 'candidateAnswers': [{'QuestionID': 650, 'Answer': 1, 'AnswerList': None,  'Info': '', 'IsImportant': 1, 'ID_CandidateQuestionType': 1, 'IsTopicQuestionType': False}]
# 'questions': [dict_keys(['Id', 'Title', 'Question', 'Info', 'ArgumentFor', 'ArgumentAgainst', 'OptionalTitle', 'OptionalQuestion', 'OptionalInfo', 'OptionalArgumentFor', 'OptionalArgumentAgainst', 'ID_CandidateQuestionType', 'QuestionListData', 'IsTopicQuestionType', 'RelatedQuestionId'])

dr_data = []
for dob in alt_dr:
    try:
        struk = {}
        struk['navn'] = dob['candidate']['Firstname'] + ' ' + dob['candidate']['LastName']
        struk['parti'] = dob['candidate']['LinedUpForParty']
        struk['parti_bogstav'] = dob['candidate']['LinedUpForPartyCode']
        struk['kreds'] = dob['candidate']['LineUps'][0]['lineUpName']

        persvar = pd.DataFrame(dob['candidateAnswers'])[['QuestionID', 'Answer']]
        persvar["QuestionID"] = "DR" + persvar["QuestionID"].astype(str)
        struk = struk | persvar.set_index("QuestionID")['Answer'].to_dict()
        dr_data.append(struk)
    except Exception as e:
        print(f"Fejl for kandidat {dob['candidate']['Firstname']} {dob['candidate']['LastName']}: {e}")
dr_df = pd.DataFrame(dr_data)
#dr_df = dr_df.replace(-0.25, np.nan)

Fejl for kandidat Camilla Dolberg Schmidt: "None of [Index(['QuestionID', 'Answer'], dtype='str')] are in the [columns]"
Fejl for kandidat Karin Liltorp: "None of [Index(['QuestionID', 'Answer'], dtype='str')] are in the [columns]"
Fejl for kandidat Bob Tønnesen: "None of [Index(['QuestionID', 'Answer'], dtype='str')] are in the [columns]"
Fejl for kandidat Bodil Helbo: "None of [Index(['QuestionID', 'Answer'], dtype='str')] are in the [columns]"
Fejl for kandidat Maria Dalum: "None of [Index(['QuestionID', 'Answer'], dtype='str')] are in the [columns]"
Fejl for kandidat Simon Jensen: "None of [Index(['QuestionID', 'Answer'], dtype='str')] are in the [columns]"
Fejl for kandidat Signe Wenneberg: "None of [Index(['QuestionID', 'Answer'], dtype='str')] are in the [columns]"
Fejl for kandidat Jesper Callesen: "None of [Index(['QuestionID', 'Answer'], dtype='str')] are in the [columns]"
Fejl for kandidat Jakob Jensen: "None of [Index(['QuestionID', 'Answer'], dtype='str')] are in the [colu

In [54]:
dr_df

,navn,parti,parti_bogstav,kreds,DR1270,DR1275,DR1277,DR1278,DR1279,DR1280,...,DR1295,DR1298,DR1299,DR1300,DR1301,DR1304,DR1305,DR1306,DR1307,DR1308
0,Stine Larsen,Alternativet,Å,Fyns Storkreds,5,4,5,2,2,1,...,1,1,4,1,2,4,5,4,4,1
1,Martin Kjærulf,Alternativet,Å,Fyns Storkreds,4,1,5,2,2,1,...,1,1,5,1,1,5,5,5,4,2
2,Ole Michael Dupont Kofod,Alternativet,Å,Fyns Storkreds,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Jacob Holm,Alternativet,Å,Fyns Storkreds,5,1,5,1,2,1,...,4,1,1,2,1,5,5,5,4,1
4,Nana Løcke Buttenschøn,Alternativet,Å,Københavns Omegns Storkreds,5,4,5,4,4,2,...,5,4,2,4,1,5,5,4,2,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
561,Per Husted,Socialdemokratiet,A,Nordjyllands Storkreds,4,2,4,2,4,2,...,4,2,2,4,2,2,4,2,2,4
562,Peder Key Kristiansen,Socialdemokratiet,A,Nordjyllands Storkreds,4,1,4,2,4,2,...,2,2,2,1,2,4,4,2,4,2
563,Kiki Bille Bach,Socialdemokratiet,A,Nordjyllands Storkreds,2,2,4,2,4,2,...,4,2,2,4,1,2,2,4,2,4
564,Flemming Møller Mortensen,Socialdemokratiet,A,Nordjyllands Storkreds,2,1,5,2,4,2,...,4,1,2,4,2,4,2,2,2,4


In [35]:
len(alt_dr)-len(dr_df)
# hmm 229 der ikke har svaret? Det er alligevel ret mange...hmmm

229

In [37]:
print(len(q), len(dr_df))
# ok ca lige mange har svaret på tv2 som på dr. Så er det måske alligevel ikke helt forkert
# Lad os prøve at merge så

575 566


In [38]:
# sikre at alle bruger samme partinavne
a = set(q.parti.str.lower().unique())
b = set(dr_df.parti.str.lower().unique())
with open('parti_map.txt', 'wt') as f:
    for parti in a.union(b):
        f.write(parti + "\n")

In [64]:
# hm ok vi skal vist lige standardisere partinavne lidt først
parti_map = {
    "venstre, danmarks liberale parti": "venstre",
    "danmarksdemokraterne ‒ inger støjberg": "danmarksdemokraterne",
    "danmarksdemokraterne": "danmarksdemokraterne",
    "venstre": "venstre",
    "sf - socialistisk folkeparti": "socialistisk folkeparti",
    "socialistisk folkeparti": "socialistisk folkeparti",
    "enhedslisten": "enhedslisten",
    "det konservative folkeparti": "det konservative folkeparti",
    "borgernes parti - lars boje mathiesen": "borgernes parti",
    "dansk folkeparti": "dansk folkeparti",
    "radikale venstre": "radikale venstre",
    "socialdemokratiet": "socialdemokratiet",
    "moderaterne": "moderaterne",
    "alternativet": "alternativet",
    "enhedslisten – de rød-grønne": "enhedslisten",
    "liberal alliance": "liberal alliance",
    "borgernes parti": "borgernes parti"
}
dr_df['parti'] = dr_df['parti'].str.lower().map(parti_map)
q['parti'] = q['parti'].str.lower().map(parti_map)

In [65]:
# ensartning af kredsnavne
a = set(q.kreds.str.lower().unique())
b = set(dr_df.kreds.str.lower().unique())
with open('kreds_map.txt', 'wt') as f:
    for kreds in a.union(b):
        f.write(kreds + "\n")

In [66]:
# de er fino

In [67]:
q['kreds'] = q.kreds.str.lower()
dr_df['kreds'] = dr_df.kreds.str.lower()

In [68]:
q[['navn', 'parti', 'kreds']].value_counts()

navn                    parti                    kreds                
Asta Kofod              enhedslisten             bornholms storkreds      1
Sebastian Bloch Jensen  socialistisk folkeparti  bornholms storkreds      1
Sofie Christiansen      socialistisk folkeparti  bornholms storkreds      1
Birthe T Bredo          venstre                  bornholms storkreds      1
Niklas Stenbye          venstre                  bornholms storkreds      1
                                                                         ..
Camilla Fabricius       socialdemokratiet        østjyllands storkreds    1
Dorthe Hindborg         socialdemokratiet        østjyllands storkreds    1
Leif Lahn Jensen        socialdemokratiet        østjyllands storkreds    1
Malte Larsen            socialdemokratiet        østjyllands storkreds    1
Thomas Monberg          socialdemokratiet        østjyllands storkreds    1
Name: count, Length: 575, dtype: int64

In [69]:
dr_df[['navn', 'parti', 'kreds']].value_counts()

navn                       parti              kreds                      
Stine Larsen               alternativet       fyns storkreds                 1
Martin Kjærulf             alternativet       fyns storkreds                 1
Ole Michael Dupont Kofod   alternativet       fyns storkreds                 1
Jacob Holm                 alternativet       fyns storkreds                 1
Nana Løcke Buttenschøn     alternativet       københavns omegns storkreds    1
                                                                            ..
Per Husted                 socialdemokratiet  nordjyllands storkreds         1
Peder Key Kristiansen      socialdemokratiet  nordjyllands storkreds         1
Kiki Bille Bach            socialdemokratiet  nordjyllands storkreds         1
Flemming Møller Mortensen  socialdemokratiet  nordjyllands storkreds         1
Ane Halsboe-Jørgensen      socialdemokratiet  nordjyllands storkreds         1
Name: count, Length: 566, dtype: int64

In [70]:
# godt så skal vi lave en samlet dataframe
# vi vil kun beholde de rows hvor vi har data fra både dr og tv2
# så vi gennemgår bare DRs datafrane også ser vi om vi kan finde et match i tv2
samlet_df = []
dr_sprg_columns = dr_df.columns[dr_df.columns.str.contains("DR")]
for i, row in dr_df.iterrows():
    # kig i tv2 data efter dr folkene.
    nemt = q[(q.navn == row.navn) & (q.kreds == row.kreds) & (q.parti == row.parti)]
    if len(nemt) == 1:
        # perfekt match
        # Tilføj tv2 data til row
        samlet_df.append(pd.concat([row[dr_sprg_columns], nemt.iloc[0]]))
    elif len(nemt) == 0:
        # ikke perfekt match
        # kan skyldes at personen ikke har svaret TV2 eller at navnet er skrevet forskelligt
        måske = q[(q.navn.str.startswith(row.navn.split(" ")[0])) & (q.navn.str.endswith(row.navn.split(" ")[-1])) & (q.parti == row.parti) & (q.kreds == row.kreds)]
        if len(måske) == 1:
            # Så er det nok bare et manglende mellemnavn
            samlet_df.append(pd.concat([row[dr_sprg_columns], måske.iloc[0]]))
        elif len(måske) == 0:
            # så bliver det svært. Nok bare nogen der ikke har svaret på begge. (eller den der ene der kalder sig "Fornavn Mellemnavn" i DR og "Fornavn Mellemnavn Efternavn" i TV2 men det gider jeg altså ikke)
            pass
        else:
            # hmm flere muligheder?
            print("flere mulige", row, måske)
            break

    else:
        print("noget helt galg", nemt)
samlet_df = pd.DataFrame(samlet_df)

In [71]:
len(samlet_df)

449

In [41]:
# 449 kandidater er med i min test

In [76]:
#Er der nogen NA værdier?
samlet_df.isna().any().any()
#samlet_df[samlet_df.isna().T.any()]
# nej

np.False_

In [77]:
# udfyld manglende svar med partiets gennemsnit
#dk_spg = list(q.columns[q.columns.str.startswith("tv2")]) + list(dr_sprg_columns)
#parti_linjen = round(samlet_df.groupby('parti')[dk_spg].mean()*4)/4  # gennemsnitssvaret per parti, rundet til 1-5 (0-4) svarmulighederne

#def fald_ind(x):
#    x.loc[x[x.isna()].index] = parti_linjen.loc[x.parti, x[x.isna()].index]
#    return x

#samlet_df_final = samlet_df.apply(fald_ind, axis=1)



In [78]:
# hvis der stadig er nogen der mangler (f.eks. fordi der var 0 i partiet der havde svaret), så giver vi dem det globale svar
#genmenet_svar = samlet_df_final[dk_spg].mean()
#def alm_svar(x):
#    x.loc[x[x.isna()].index] = genmenet_svar.loc[x[x.isna()].index]
#    return x
#
#samlet_df_final_final = samlet_df_final.apply(alm_svar, axis=1)

In [79]:
samlet_df.to_feather("2026_FV_Lasse_data.feather")